# Q2 · Broadband LFP power
Run `q2_lfp_power.py` and `q2_lfp_power_stats.R` before executing this notebook. The curves for each mouse and area are pointwise medians across valid channels. The across-mouse curves are pointwise medians of those spectra, giving each mouse equal weight. The broadband power plot uses each mouse’s own spectrum integrated from 1 to 100 Hz.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from matplotlib.lines import Line2D

configured_output = os.environ.get("Q2_OUTPUT_DIR")
if configured_output:
    output = Path(configured_output)
else:
    cwd = Path.cwd()
    root = next((p for p in (cwd, *cwd.parents)
                 if (p / "code/questions/q2_lfp_power.py").is_file()), None)
    if root is None:
        raise FileNotFoundError("Open this notebook within the repository or set Q2_OUTPUT_DIR.")
    output = root / "results" / "q2"

config = json.loads((output / "q2_config.json").read_text(encoding="utf-8"))
spectra = pd.read_csv(output / "q2_mouse_area_psd.csv")
median_spectra = pd.read_csv(output / "q2_area_median_psd.csv")
power = pd.read_csv(output / "q2_mouse_area_power.csv")
means = pd.read_csv(output / "q2_stats_means.csv").sort_values("brain_area")
mice = sorted(spectra.mouse_id.unique())
areas = config["areas"]
band = config["power_band_hz"]
first_hz = config["welch"]["first_frequency_hz"]
last_hz = config["welch"]["last_frequency_hz"]
colors = {1: "#0072B2", 2: "#D55E00", 3: "#00856A"}
power_unit = config["power_unit"].replace("uV", "µV").replace("^2", "²")
psd_unit = f"{power_unit}/Hz"
spectra_by_mouse_area = spectra.groupby(["mouse_id", "brain_area"])
medians_by_area = median_spectra.groupby("brain_area")

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 10})

def format_spectrum(ax):
    ax.set(xscale="log", yscale="log", xlim=(first_hz, last_hz))
    ax.axvspan(*band, color="0.5", alpha=0.08, zorder=0)
    ax.set_xticks([0.3, 1, 3, 10, 30, 100], labels=["0.3", "1", "3", "10", "30", "100"])

def save_and_show(fig, filename):
    path = output / filename
    fig.savefig(path, dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    display(Image(filename=str(path)))

## Mouse spectra by area
Each thin line represents one mouse. The bold line is the across-mouse median at each frequency. Shading marks the 1–100 Hz integration band.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.8), sharex=True, sharey=True,
                         layout="constrained")
for ax, area in zip(axes, areas):
    for mouse in mice:
        trace = spectra_by_mouse_area.get_group((mouse, area))
        ax.plot(trace.frequency_hz, trace.psd, color=colors[area], alpha=0.29, lw=0.7)
    median = medians_by_area.get_group(area)
    ax.plot(median.frequency_hz, median.median_psd, color=colors[area], lw=2.3,
            label=f"Median of {len(mice)} mice")
    format_spectrum(ax)
    ax.set(title=f"Area {area}", xlabel="Frequency (Hz)")
    ax.legend(frameon=False, loc="upper right")
axes[0].set_ylabel(f"PSD ({psd_unit})")
save_and_show(fig, "q2_psd_mice_by_area.png")

## Area spectra within each mouse
All panels use the same frequency and power-density scales.

In [ ]:
n_cols = 6
n_rows = (len(mice) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(13.4, 2.35 * n_rows + 0.5),
                         sharex=True, sharey=True, layout="constrained", squeeze=False)
for ax, mouse in zip(axes.flat, mice):
    for area in areas:
        trace = spectra_by_mouse_area.get_group((mouse, area))
        ax.plot(trace.frequency_hz, trace.psd, color=colors[area], lw=1.15)
    format_spectrum(ax)
    ax.set_title(f"Mouse {mouse}")
for ax in list(axes.flat)[len(mice):]:
    ax.set_visible(False)

fig.supxlabel("Frequency (Hz)")
fig.supylabel(f"PSD ({psd_unit})")
handles = [Line2D([0], [0], color=colors[area], label=f"Area {area}") for area in areas]
fig.legend(handles=handles, loc="outside upper center", ncols=len(areas), frameon=False)
save_and_show(fig, "q2_psd_areas_by_mouse.png")

## Median PSD across mice
Overlay of the three area medians from the first figure.

In [ ]:
fig, ax = plt.subplots(figsize=(6.3, 4.1), layout="constrained")
for area in areas:
    median = medians_by_area.get_group(area)
    ax.plot(median.frequency_hz, median.median_psd, color=colors[area], lw=2,
            label=f"Area {area}")
format_spectrum(ax)
ax.set(xlabel="Frequency (Hz)", ylabel=f"PSD ({psd_unit})",
       title=f"Median LFP spectra across {len(mice)} mice")
ax.legend(frameon=False)
save_and_show(fig, "q2_psd_area_medians.png")

## Integrated 1–100 Hz power
Connecting lines link areas recorded in the same mouse. The right panel shows the R model’s geometric means and pointwise 95% confidence intervals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.2, 4.0), sharey=True, layout="constrained")
wide = power.pivot(index="mouse_id", columns="brain_area", values="broadband_power")
for _, row in wide.iterrows():
    axes[0].plot(areas, row.loc[areas], color="0.55", alpha=0.42, lw=0.9, zorder=1)
for area in areas:
    axes[0].scatter(np.full(len(wide), area), wide[area], color=colors[area],
                    s=20, alpha=0.73, zorder=2)

for row in means.itertuples(index=False):
    interval = [[row.geometric_mean - row.lower_95], [row.upper_95 - row.geometric_mean]]
    axes[1].errorbar(row.brain_area, row.geometric_mean, yerr=interval, fmt="o", capsize=5,
                     color=colors[row.brain_area], lw=1.9)

for ax, title in zip(axes, ["Paired mouse observations", "Geometric means and 95% CIs"]):
    ax.set(yscale="log", title=title, xlim=(0.75, 3.25))
    ax.set_xticks(areas, labels=[f"Area {area}" for area in areas])
axes[0].set_ylabel(f"Integrated 1–100 Hz power ({power_unit})")
save_and_show(fig, "q2_broadband_power.png")
means[["brain_area", "n_mice", "geometric_mean", "lower_95", "upper_95", "power_unit"]]